In [1]:
import pandas as pd
import pickle
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn_quantile import RandomForestQuantileRegressor
from tqdm.auto import tqdm
import CRPS.CRPS as pscore


import multiprocessing as mp
mp.set_start_method('spawn')


import sys
sys.path.append('../../../TaskExecutionTimeMining/')
from quantile_regression import QuantileRegression


sys.path.append('../../../Evaluation/')
import conduct_evaluation
from normal_evaluation.quantile_regression_evaluation import *
from normal_evaluation.normal_evaluation import SampleOutcomes_Normal

get_pscores = lambda likelihoods : [pscore(likelihoods[1][i], likelihoods[2][k][3]).compute()[0] for i, k in enumerate(list(likelihoods[0].keys()))]


In [2]:
with open('./quantile_regression_models.pkl', 'rb') as f:
    quantile_regression_models = pickle.load(f)

In [3]:
with open('../../transformed_event_logs/Helpdesk_test.pickle', 'rb') as f:
    test_data = pickle.load(f)

test_data['Case ID'] = test_data['Case ID'].astype(str)
test_data['case:concept:name'] = test_data['Case ID']
test_data['time:timestamp_start'] = test_data['Complete Timestamp_start']
test_data['time:timestamp_complete'] = test_data['Complete Timestamp_complete']

activity_count = [
    'Assign seriousness',
    'Closed',
    'Create SW anomaly',
    'DUPLICATE',
    'INVALID',
    'Insert ticket',
    'RESOLVED',
    'Require upgrade',
    'Resolve SW anomaly',
    'Resolve ticket',
    'Schedule intervention',
    'Take in charge ticket',
    'VERIFIED',
    'Wait'
 ]

resource_count = [
 'Value 1',
 'Value 10',
 'Value 11',
 'Value 12',
 'Value 13',
 'Value 14',
 'Value 15',
 'Value 16',
 'Value 17',
 'Value 18',
 'Value 19',
 'Value 2',
 'Value 20',
 'Value 21',
 'Value 22',
 'Value 3',
 'Value 4',
 'Value 5',
 'Value 6',
 'Value 7',
 'Value 8',
 'Value 9',
]

ii1 = [
   'intercase_n_1__Assign seriousness',
 'intercase_n_1__Closed',
 'intercase_n_1__Create SW anomaly',
 'intercase_n_1__DUPLICATE',
 'intercase_n_1__INVALID',
 'intercase_n_1__Insert ticket',
 'intercase_n_1__RESOLVED',
 'intercase_n_1__Require upgrade',
 'intercase_n_1__Resolve SW anomaly',
 'intercase_n_1__Resolve ticket',
 'intercase_n_1__Schedule intervention',
 'intercase_n_1__Take in charge ticket',
 'intercase_n_1__VERIFIED',
 'intercase_n_1__Wait', 
]

ii3 = [
    'intercase_n_3__Assign seriousness',
 'intercase_n_3__Assign seriousness_Assign seriousness',
 'intercase_n_3__Assign seriousness_Assign seriousness_Assign seriousness',
 'intercase_n_3__Assign seriousness_Assign seriousness_Resolve ticket',
 'intercase_n_3__Assign seriousness_Assign seriousness_Take in charge ticket',
 'intercase_n_3__Assign seriousness_Assign seriousness_Wait',
 'intercase_n_3__Assign seriousness_Create SW anomaly',
 'intercase_n_3__Assign seriousness_Create SW anomaly_Create SW anomaly',
 'intercase_n_3__Assign seriousness_Create SW anomaly_Require upgrade',
 'intercase_n_3__Assign seriousness_Create SW anomaly_Take in charge ticket',
 'intercase_n_3__Assign seriousness_Require upgrade',
 'intercase_n_3__Assign seriousness_Require upgrade_Require upgrade',
 'intercase_n_3__Assign seriousness_Require upgrade_Resolve ticket',
 'intercase_n_3__Assign seriousness_Resolve ticket',
 'intercase_n_3__Assign seriousness_Resolve ticket_Closed',
 'intercase_n_3__Assign seriousness_Resolve ticket_Resolve ticket',
 'intercase_n_3__Assign seriousness_Resolve ticket_Take in charge ticket',
 'intercase_n_3__Assign seriousness_Resolve ticket_Wait',
 'intercase_n_3__Assign seriousness_Take in charge ticket',
 'intercase_n_3__Assign seriousness_Take in charge ticket_Assign seriousness',
 'intercase_n_3__Assign seriousness_Take in charge ticket_Create SW anomaly',
 'intercase_n_3__Assign seriousness_Take in charge ticket_Require upgrade',
 'intercase_n_3__Assign seriousness_Take in charge ticket_Resolve SW anomaly',
 'intercase_n_3__Assign seriousness_Take in charge ticket_Resolve ticket',
 'intercase_n_3__Assign seriousness_Take in charge ticket_Schedule intervention',
 'intercase_n_3__Assign seriousness_Take in charge ticket_Take in charge ticket',
 'intercase_n_3__Assign seriousness_Take in charge ticket_Wait',
 'intercase_n_3__Assign seriousness_Wait',
 'intercase_n_3__Assign seriousness_Wait_Assign seriousness',
 'intercase_n_3__Assign seriousness_Wait_Resolve ticket',
 'intercase_n_3__Assign seriousness_Wait_Take in charge ticket',
 'intercase_n_3__Assign seriousness_Wait_Wait',
 'intercase_n_3__Closed_Take in charge ticket_Resolve ticket',
 'intercase_n_3__Create SW anomaly',
 'intercase_n_3__Create SW anomaly_Create SW anomaly_Resolve SW anomaly',
 'intercase_n_3__Create SW anomaly_Create SW anomaly_Resolve ticket',
 'intercase_n_3__Create SW anomaly_Require upgrade_Require upgrade',
 'intercase_n_3__Create SW anomaly_Require upgrade_Resolve ticket',
 'intercase_n_3__Create SW anomaly_Require upgrade_VERIFIED',
 'intercase_n_3__Create SW anomaly_Resolve SW anomaly',
 'intercase_n_3__Create SW anomaly_Resolve SW anomaly_Resolve SW anomaly',
 'intercase_n_3__Create SW anomaly_Resolve SW anomaly_Resolve ticket',
 'intercase_n_3__Create SW anomaly_Resolve ticket_RESOLVED',
 'intercase_n_3__Create SW anomaly_Resolve ticket_Take in charge ticket',
 'intercase_n_3__Create SW anomaly_Take in charge ticket_Create SW anomaly',
 'intercase_n_3__Create SW anomaly_Take in charge ticket_Wait',
 'intercase_n_3__Insert ticket',
 'intercase_n_3__Insert ticket_Assign seriousness',
 'intercase_n_3__Insert ticket_Assign seriousness_Assign seriousness',
 'intercase_n_3__Insert ticket_Assign seriousness_Resolve ticket',
 'intercase_n_3__Insert ticket_Assign seriousness_Take in charge ticket',
 'intercase_n_3__Insert ticket_Take in charge ticket',
 'intercase_n_3__Insert ticket_Take in charge ticket_Resolve ticket',
 'intercase_n_3__Insert ticket_Wait',
 'intercase_n_3__Insert ticket_Wait_Wait',
 'intercase_n_3__RESOLVED_INVALID_Closed',
 'intercase_n_3__RESOLVED_INVALID_VERIFIED',
 'intercase_n_3__Require upgrade_Create SW anomaly_Resolve ticket',
 'intercase_n_3__Require upgrade_Require upgrade_Create SW anomaly',
 'intercase_n_3__Require upgrade_Require upgrade_Require upgrade',
 'intercase_n_3__Require upgrade_Require upgrade_Resolve ticket',
 'intercase_n_3__Require upgrade_Require upgrade_Take in charge ticket',
 'intercase_n_3__Require upgrade_Resolve ticket_Resolve ticket',
 'intercase_n_3__Require upgrade_Take in charge ticket_Resolve ticket',
 'intercase_n_3__Require upgrade_Take in charge ticket_Wait',
 'intercase_n_3__Require upgrade_VERIFIED_DUPLICATE',
 'intercase_n_3__Require upgrade_Wait_Resolve ticket',
 'intercase_n_3__Resolve SW anomaly_Require upgrade_Create SW anomaly',
 'intercase_n_3__Resolve SW anomaly_Require upgrade_Resolve ticket',
 'intercase_n_3__Resolve SW anomaly_Resolve SW anomaly_Require upgrade',
 'intercase_n_3__Resolve SW anomaly_Resolve SW anomaly_Resolve ticket',
 'intercase_n_3__Resolve ticket',
 'intercase_n_3__Resolve ticket_Assign seriousness_Take in charge ticket',
 'intercase_n_3__Resolve ticket_Closed_Take in charge ticket',
 'intercase_n_3__Resolve ticket_RESOLVED_INVALID',
 'intercase_n_3__Resolve ticket_Require upgrade_Take in charge ticket',
 'intercase_n_3__Resolve ticket_Resolve ticket_Require upgrade',
 'intercase_n_3__Resolve ticket_Resolve ticket_Resolve ticket',
 'intercase_n_3__Resolve ticket_Resolve ticket_Take in charge ticket',
 'intercase_n_3__Resolve ticket_Take in charge ticket',
 'intercase_n_3__Resolve ticket_Take in charge ticket_Create SW anomaly',
 'intercase_n_3__Resolve ticket_Take in charge ticket_Require upgrade',
 'intercase_n_3__Resolve ticket_Take in charge ticket_Resolve ticket',
 'intercase_n_3__Resolve ticket_Take in charge ticket_Take in charge ticket',
 'intercase_n_3__Resolve ticket_Take in charge ticket_Wait',
 'intercase_n_3__Resolve ticket_Wait_Resolve ticket',
 'intercase_n_3__Resolve ticket_Wait_Wait',
 'intercase_n_3__Schedule intervention_Take in charge ticket_Wait',
 'intercase_n_3__Take in charge ticket',
 'intercase_n_3__Take in charge ticket_Assign seriousness_Assign seriousness',
 'intercase_n_3__Take in charge ticket_Create SW anomaly_Create SW anomaly',
 'intercase_n_3__Take in charge ticket_Create SW anomaly_Require upgrade',
 'intercase_n_3__Take in charge ticket_Create SW anomaly_Resolve SW anomaly',
 'intercase_n_3__Take in charge ticket_Create SW anomaly_Resolve ticket',
 'intercase_n_3__Take in charge ticket_Create SW anomaly_Take in charge ticket',
 'intercase_n_3__Take in charge ticket_Require upgrade_Create SW anomaly',
 'intercase_n_3__Take in charge ticket_Require upgrade_Require upgrade',
 'intercase_n_3__Take in charge ticket_Require upgrade_Resolve ticket',
 'intercase_n_3__Take in charge ticket_Require upgrade_Take in charge ticket',
 'intercase_n_3__Take in charge ticket_Require upgrade_Wait',
 'intercase_n_3__Take in charge ticket_Resolve SW anomaly_Resolve ticket',
 'intercase_n_3__Take in charge ticket_Resolve ticket',
 'intercase_n_3__Take in charge ticket_Resolve ticket_Assign seriousness',
 'intercase_n_3__Take in charge ticket_Resolve ticket_Closed',
 'intercase_n_3__Take in charge ticket_Resolve ticket_Resolve ticket',
 'intercase_n_3__Take in charge ticket_Resolve ticket_Take in charge ticket',
 'intercase_n_3__Take in charge ticket_Resolve ticket_Wait',
 'intercase_n_3__Take in charge ticket_Schedule intervention_Resolve ticket',
 'intercase_n_3__Take in charge ticket_Schedule intervention_Take in charge ticket',
 'intercase_n_3__Take in charge ticket_Take in charge ticket',
 'intercase_n_3__Take in charge ticket_Take in charge ticket_Create SW anomaly',
 'intercase_n_3__Take in charge ticket_Take in charge ticket_Require upgrade',
 'intercase_n_3__Take in charge ticket_Take in charge ticket_Resolve ticket',
 'intercase_n_3__Take in charge ticket_Take in charge ticket_Schedule intervention',
 'intercase_n_3__Take in charge ticket_Take in charge ticket_Take in charge ticket',
 'intercase_n_3__Take in charge ticket_Take in charge ticket_Wait',
 'intercase_n_3__Take in charge ticket_Wait',
 'intercase_n_3__Take in charge ticket_Wait_Create SW anomaly',
 'intercase_n_3__Take in charge ticket_Wait_Require upgrade',
 'intercase_n_3__Take in charge ticket_Wait_Resolve ticket',
 'intercase_n_3__Take in charge ticket_Wait_Take in charge ticket',
 'intercase_n_3__Take in charge ticket_Wait_Wait',
 'intercase_n_3__VERIFIED_DUPLICATE_Resolve ticket',
 'intercase_n_3__Wait',
 'intercase_n_3__Wait_Assign seriousness_Assign seriousness',
 'intercase_n_3__Wait_Assign seriousness_Take in charge ticket',
 'intercase_n_3__Wait_Create SW anomaly_Require upgrade',
 'intercase_n_3__Wait_Create SW anomaly_Resolve ticket',
 'intercase_n_3__Wait_Require upgrade_Require upgrade',
 'intercase_n_3__Wait_Require upgrade_Resolve ticket',
 'intercase_n_3__Wait_Require upgrade_Wait',
 'intercase_n_3__Wait_Resolve ticket',
 'intercase_n_3__Wait_Resolve ticket_Resolve ticket',
 'intercase_n_3__Wait_Resolve ticket_Take in charge ticket',
 'intercase_n_3__Wait_Resolve ticket_Wait',
 'intercase_n_3__Wait_Take in charge ticket_Create SW anomaly',
 'intercase_n_3__Wait_Take in charge ticket_Require upgrade',
 'intercase_n_3__Wait_Take in charge ticket_Resolve ticket',
 'intercase_n_3__Wait_Take in charge ticket_Take in charge ticket',
 'intercase_n_3__Wait_Take in charge ticket_Wait',
 'intercase_n_3__Wait_Wait_Create SW anomaly',
 'intercase_n_3__Wait_Wait_Resolve ticket',
 'intercase_n_3__Wait_Wait_Take in charge ticket',
 'intercase_n_3__Wait_Wait_Wait'
]

In [4]:
n_processes = 32
batch_size = 16
N = 1000

In [5]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['A'], SampleOutcomes_QuantileRegression_A, {
                                                        'case_id_key' : 'Case_ID',
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : 'Resource_start',
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

In [6]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-4.270196181218299742876890071')

In [7]:
np.mean(get_pscores(likelihoods_A))

np.float64(895420.7731797103)

In [8]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['R'], SampleOutcomes_QuantileRegression_R, {
                                                        'case_id_key' : 'Case_ID',
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : 'Resource_start',
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

In [9]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-5.070466202501707147707151074')

In [10]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-5.070466202501707147707151074')

In [11]:
np.mean(get_pscores(likelihoods_A))

np.float64(2064663.3006590907)

In [12]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['AR'], SampleOutcomes_QuantileRegression_AR, {
                                                        'case_id_key' : 'Case_ID',
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : 'Resource_start',
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

In [13]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-4.361262745812443947092125897')

In [14]:
np.mean(get_pscores(likelihoods_A))

np.float64(860194.6363931719)

In [15]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['ARS'], SampleOutcomes_QuantileRegression_ARS, {
                                                        'case_id_key' : 'Case_ID',
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : 'Resource_start',
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

In [16]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-5.578823303742160331957602688')

In [17]:
np.mean(get_pscores(likelihoods_A))

np.float64(954883.8678924579)

In [18]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['RSAC'], SampleOutcomes_QuantileRegression_ARSAC, {
                                                        'case_id_key' : 'Case_ID',
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : 'Resource_start',
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

KeyError: 'RSAC'

In [19]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-5.578823303742160331957602688')

In [20]:
np.mean(get_pscores(likelihoods_A))

np.float64(954883.8678924579)

In [21]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['ARSRC'], SampleOutcomes_QuantileRegression_ARSRC, {
                                                        'case_id_key' : 'Case_ID',
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : 'Resource_start',
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

KeyError: 'ARSRC'

In [22]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-5.578823303742160331957602688')

In [23]:
np.mean(get_pscores(likelihoods_A))

np.float64(954883.8678924579)

In [24]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['ARSACRC'], SampleOutcomes_QuantileRegression_ARSACRC, {
                                                        'case_id_key' : 'Case_ID',
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : 'Resource_start',
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

In [25]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-13.72097744981843926540341823')

In [26]:
np.mean(get_pscores(likelihoods_A))

np.float64(1046771.5442518939)

In [27]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['ARSD'], SampleOutcomes_QuantileRegression_ARSD, {
                                                        'case_id_key' : 'Case_ID',
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : 'Resource_start',
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

In [28]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-4.752405410275564066857724962')

In [29]:
np.mean(get_pscores(likelihoods_A))

np.float64(1024166.6946035597)

In [30]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['ARSDACRC'], SampleOutcomes_QuantileRegression_ARSDACRC, {
                                                        'case_id_key' : 'Case_ID',
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : 'Resource_start',
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

In [31]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-4.486080668964608347128472083')

In [32]:
np.mean(get_pscores(likelihoods_A))

np.float64(1075718.4790342334)

In [33]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['ARSDACRCII1'],
                                                   SampleOutcomes_QuantileRegression_ARSDACRCII, {
                                                        'case_id_key' : 'Case_ID',
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : 'Resource_start',
                                                        'inter_instance_column_names' : ii1
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)


In [34]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-4.842689786552707856416412152')

In [35]:
np.mean(get_pscores(likelihoods_A))

np.float64(1490172.113958698)

In [36]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['ARSDACRCII3'],
                                                   SampleOutcomes_QuantileRegression_ARSDACRCII, {
                                                        'case_id_key' : 'Case_ID',
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : 'Resource_start',
                                                        'inter_instance_column_names' : ii3
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

In [37]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-9.425060887538019139998466660')

In [38]:
np.mean(get_pscores(likelihoods_A))

np.float64(1228531.0948394053)

In [39]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['ARSDII1'],
                                                   SampleOutcomes_QuantileRegression_ARSDII, {
                                                        'case_id_key' : 'Case_ID',
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : 'Resource_start',
                                                        'inter_instance_column_names' : ii1
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

In [40]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-4.970929024705667343637855644')

In [41]:
np.mean(get_pscores(likelihoods_A))

np.float64(1601200.7932891787)

In [42]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['ARSDII3'],
                                                   SampleOutcomes_QuantileRegression_ARSDII, {
                                                        'case_id_key' : 'Case_ID',
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : 'Resource_start',
                                                        'inter_instance_column_names' : ii3
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

In [43]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-6.768691646450240189171233593')

In [44]:
np.mean(get_pscores(likelihoods_A))

np.float64(1349307.0248745482)